# VADMamba++

## Background

### Unsupervised Video Anomaly Detection

Unlike supervised datasets (e.g., UCF-Crime or XD-Violence), unsupervised VAD is trained **only with normal videos**.

Therefore, the objective is not to classify anomalies directly, but to **learn the distribution of normal events** and detect deviations during inference.

Most unsupervised methods achieve this through three proxy paradigms:

- Reconstruction
- Prediction
- Feature Embedding

---

## Evolution of Detection Architectures

The paper groups recent unsupervised VAD methods into three architectural families.

### CNN-based

- Autoencoders
- GANs
- Diffusion models

Strengths:

- Good local feature extraction.
- Efficient inference.

Limitations:

- Limited receptive field.
- Difficult to model long-range temporal dependencies.

---

### Transformer-based

Transformers solve the locality problem through self-attention.

Advantages:

- Global spatial context.
- Long-range temporal modeling.

Limitations:

- High computational cost.
- Quadratic complexity.

---

### Mamba-based

Mamba introduces State Space Models to obtain:

- Linear complexity.
- Long-range dependency modeling.
- Better efficiency than Transformers.

According to the authors, the progression of Mamba-based VAD is:

- **VADMamba (2025):** First VAD model using Mamba with hybrid proxy tasks.
- **STNMamba (2024/2025):** Introduces spatial-temporal normality learning with RGB-difference features and multiple memory modules.
- **VADMamba++:** Simplifies the paradigm while improving both efficiency and accuracy.

---

# Motivation

The authors argue that existing methods improve accuracy by introducing additional complexity.

Typical improvements require:

- Optical Flow
- Difference images
- Gaussian noise
- Random masking
- Multiple proxy tasks
- Multi-input pipelines

These techniques improve performance but:

- increase preprocessing cost;
- reduce inference speed;
- complicate deployment.

Furthermore, conventional reconstruction methods follow an **RGB → RGB** paradigm.

Although this allows the network to reproduce normal appearance, it also encourages reconstructing the colors of anomalous objects, making anomalies less distinguishable.

---

# Main Idea

Instead of reconstructing RGB from RGB, VADMamba++ formulates anomaly detection as a **reasoning problem**.

```text
Gray Frame
      │
      ▼
Learn scene structure
      │
Infer plausible colors
      │
      ▼
RGB Prediction
```

This is called the **Gray-to-RGB (G2R) paradigm**.

The model receives only grayscale frames and must infer the corresponding RGB frame.

The hypothesis is that anomalies become easier to detect because the model simultaneously fails to reconstruct:

- structural geometry;
- chromatic appearance.

Thus, anomalies generate two complementary inconsistencies instead of only appearance errors.

---

# Differences with VADMamba

The paper identifies three major evolutions.

| VADMamba | VADMamba++ |
|----------|------------|
| Multi-input | Single-input |
| Multi-task | Single-task |
| Inter-task fusion | Intra-task fusion |
| RGB → RGB | Gray → RGB |
| Uses Optical Flow | No Optical Flow |

The objective is to simplify the entire pipeline while maintaining or improving detection performance.

---

# Proposed Architecture

VADMamba++ consists of three main components.

## 1. TMC Encoder

The encoder combines three complementary modules:

- Transformer
- Mamba (BiSS2D)
- CNN (Dual Pre-activation)

Each one contributes differently:

| Module | Purpose |
|---------|---------|
| Transformer | Global context modeling |
| Mamba | Long-range temporal dependencies |
| CNN | Local spatial refinement |

The best ordering found experimentally is:

```text
Transformer
      ↓
Mamba
      ↓
CNN
```

This coarse-to-fine processing achieved the highest AUC.

---

## 2. MC Decoder

Unlike the encoder, the decoder removes the Transformer.

Reason:

Only a single future frame must be reconstructed.

Therefore:

- temporal reasoning is already completed in the encoder;
- the decoder only restores spatial details.

This produces a lighter architecture.

---

## 3. Vector Quantization (VQ)

The encoder output is quantized using a VQ bottleneck.

The quantized representation is used:

- for reconstruction;
- as an implicit anomaly representation during inference.

---

# Hybrid Modeling

Rather than relying exclusively on one architecture, VADMamba++ combines:

- Transformer
- Mamba
- CNN

The authors argue that these architectures provide complementary capabilities.

---

# Anomaly Score

Unlike traditional methods that rely only on reconstruction error, VADMamba++ combines two scores.

### Explicit Score

Prediction error:

```text
Ground Truth RGB
        -
Predicted RGB
```

Measures visible reconstruction error.

---

### Implicit Score

Distance between:

- latent features;
- quantized features.

This measures semantic inconsistency inside the latent space.

---

### Final Score

The final anomaly score is:

```text
Explicit Error
        +
Weighted Implicit Error
```

The authors call this **Intra-task Fusion Scoring**.

---

# Main Experimental Results

## Single-task Comparison

| Method | Avenue | SHT | Ped2 | FPS |
|---------|-------:|----:|------:|----:|
| STNMamba | 89.0 | 74.9 | 98.0 | 40 |
| FastAno+ | 87.8 | 75.2 | 98.1 | 130 |
| VADMamba (FP) | 86.2 | 74.4 | 98.1 | **151** |
| **VADMamba++** | **91.9** | **77.1** | **99.6** | **133** |

**Observation**

Among single-task methods, VADMamba++ achieves the best overall balance between accuracy and speed.

---

## Multi-task Comparison

| Method | Tasks | Avenue | SHT | Ped2 | FPS |
|---------|------:|-------:|----:|------:|----:|
| HF2-VAD | 2 | 91.1 | 76.2 | 99.3 | 15 |
| VADMamba | 2 | 91.5 | 77.0 | 98.5 | 90 |
| SSAE | 2 | 90.2 | **80.5** | - | 10 |
| **VADMamba++** | **1** | **91.9** | **77.1** | **99.6** | **133** |

Although some multi-task methods obtain slightly higher AUC on ShanghaiTech, they require multiple proxy tasks and are substantially slower.

VADMamba++ achieves competitive performance using:

- a single proxy task;
- frame-only inputs;
- no Optical Flow.

---

# Main Contributions

- Introduces a **Gray-to-RGB reasoning paradigm** instead of conventional RGB reconstruction.
- Eliminates Optical Flow completely.
- Converts a multi-task pipeline into a single-task framework.
- Combines Transformer, Mamba, and CNN in a lightweight hybrid architecture.
- Introduces **Intra-task Fusion** by combining explicit reconstruction error with implicit latent-space error.
- Achieves one of the best accuracy-speed trade-offs among unsupervised VAD methods.

---

# Comparison with Previous Unsupervised VAD

| Generation | Main Idea | Limitation |
|------------|-----------|------------|
| CNN-based | Learn local appearance | Limited temporal modeling |
| Transformer-based | Learn global dependencies | High computational cost |
| STNMamba | Improve spatial-temporal learning with Mamba | Still focused on architectural improvements |
| **VADMamba++** | Simplify the learning paradigm (Gray→RGB, single-task, no Optical Flow) | Focuses on efficiency while maintaining SOTA performance |

## Position of VADMamba++ within Unsupervised VAD

It is important to distinguish the contribution of VADMamba++ from previous Mamba-based approaches.

- **STNMamba** focuses on **improving the architecture** by redesigning Mamba blocks (MS-VSSB and CA-VSSB) and introducing multi-level spatial-temporal interaction (STIM) to better model normal spatio-temporal patterns.

- **VADMamba++**, in contrast, focuses on **changing the learning paradigm**. Instead of proposing a fundamentally different Mamba architecture, it reformulates the proxy task itself by replacing the conventional **RGB → RGB** reconstruction with a **Gray → RGB reasoning** task. At the same time, it simplifies the overall pipeline by removing Optical Flow, eliminating multi-task learning, and requiring only frame-level inputs.

Therefore, the evolution of unsupervised VAD can be interpreted as:

| Generation | Main Idea |
|------------|-----------|
| CNN-based methods | Learn normal appearance through reconstruction or prediction. |
| Transformer-based methods | Improve global spatial-temporal modeling using self-attention. |
| STNMamba | Improve **how** the model learns normality through a better Mamba-based architecture. |
| **VADMamba++** | Improve **what** the model learns by redefining the proxy task into a simpler and more informative Gray-to-RGB reasoning problem. |

This distinction is important because STNMamba and VADMamba++ are **complementary rather than competing** approaches. STNMamba mainly contributes architectural innovations, whereas VADMamba++ contributes a new training paradigm that achieves a better balance between accuracy, inference speed, and deployment simplicity.

---

## Why Does Gray-to-RGB Make Anomalies Easier to Detect?

The core idea behind VADMamba++ is that reconstructing an RGB image from a grayscale input is **not a simple reconstruction task**, but a **reasoning task**.

Unlike conventional RGB→RGB methods, the model no longer receives color information as input. Instead, it must infer what the scene **should normally look like**, based only on its learned knowledge of normal events.

### Conventional RGB → RGB Reconstruction

Traditional reconstruction-based methods receive an RGB frame and attempt to reproduce it.

```text
RGB Frame
     │
     ▼
   Model
     │
     ▼
Reconstructed RGB
```

The anomaly score is computed as:

```text
Original RGB
      -
Reconstructed RGB
```

Since the model already has access to the original colors, it can often reproduce abnormal objects reasonably well. As a result, some anomalies generate only small reconstruction errors.

---

### Gray → RGB Reasoning

VADMamba++ removes all color information before feeding the image to the network.

```text
Gray Frame
     │
     ▼
   Model
     │
     ▼
Predicted RGB
```

Now the model must answer a much harder question:

> **"What should the normal color appearance of this scene be?"**

Instead of copying colors, the network must infer them from the structural information contained in the grayscale image.

---

## Structural Geometry

The first challenge is reconstructing the correct **geometry** of the scene.

The model has learned the shapes and layouts of normal events.

For example:

```text
Training

Walking person
```

During inference, an abnormal object appears:

```text
Ground Truth

🚲 Bicycle
```

The model may instead predict something closer to a walking person.

```text
Prediction

🚶 Person
```

This produces a **structural inconsistency**, since the reconstructed geometry does not match the real object.

---

## Chromatic Appearance

The second challenge is reconstructing plausible colors.

Because the input is grayscale, the model does not know whether an object was:

- red,
- blue,
- green,
- yellow.

It must infer the most likely colors based on the normal scenes observed during training.

If the object is abnormal, the inferred colors are also likely to be incorrect, producing a **chromatic inconsistency**.

---

## Dual Inconsistencies

Unlike RGB→RGB reconstruction, Gray→RGB reasoning introduces **two complementary sources of error**.

```text
Gray Input
      │
      ▼
Predict RGB
      │
      ▼
Is the geometry correct?
            +
Are the colors correct?
```

Therefore, anomalies simultaneously violate:

- **Structural Geometry**
  - object shape
  - spatial layout
  - scene structure

- **Chromatic Appearance**
  - color consistency
  - appearance consistency

Instead of measuring only appearance reconstruction error, VADMamba++ detects anomalies through both **structural** and **chromatic** inconsistencies.

This is why the authors state that anomalies generate **two complementary inconsistencies** rather than only appearance errors.

---

## Intuition

Suppose the model has only learned normal pedestrians.

During inference, a motorcycle appears.

### RGB → RGB

```text
Input
🏍️

↓

Reconstruction
🏍️
```

Since the model already sees the motorcycle and its colors, it may reconstruct it reasonably well, producing only a moderate reconstruction error.

### Gray → RGB

```text
Input
███
```

Now the model must infer:

- What object is this?
- What shape should it have?
- What colors should it have?

Because it has never learned motorcycles as normal events, it may reconstruct:

```text
🚶 Person
```

or

```text
🚲 Bicycle
```

with incorrect colors.

Both the **geometry** and the **appearance** become inconsistent, leading to a significantly larger anomaly score.